# Importing Libraries 

In [1]:
#importing libraries
import csv
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException, TimeoutException

# Opening MeroKirana via Chrome Beta 

In [2]:
#specifying the path to the Chrome binary
chrome_binary_path = "C:\Program Files\Google\Chrome Beta\Application\chrome.exe"

#setting the binary location in ChromeOptions
chrome_options = webdriver.ChromeOptions()
chrome_options.binary_location = chrome_binary_path

#specifying the path to ChromeDriver
chrome_driver_path = r"D:\chromedriver-win64\chromedriver.exe"

#creating a ChromeDriver service with the specified binary path
service = Service(executable_path=chrome_driver_path)

#creating the WebDriver instance with the specified options and service
driver = webdriver.Chrome(service=service, options=chrome_options)

#maximizing the window and opening the link for scraping
driver.maximize_window()
driver.get('https://merokirana.com/#/')

# Scraping the fruit prices

In [3]:
#clicking the Veg & Fruits option from navbar
veg_fruits = driver.find_element(By.XPATH, "//*[contains(text(), 'Veg & Fruits')]").click()

#selecting the fruits from dropdown to be redirected to that webpage
fruits = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-35ae95a75bda48a3-ac60b32fab01d5db']/a"))).click()

In [4]:
# Function to check if all price elements are non-empty
def are_prices_non_empty():
    price_elements = driver.find_elements(By.XPATH, "//div[@class='product-card__price']")
    return all(price_elem.text.strip() for price_elem in price_elements)

#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

# Refresh the page until all price elements are non-empty
while not are_prices_non_empty():
    driver.refresh()
    time.sleep(5)  # adding a delay to ensure the page fully reloads

In [5]:
#collecting the name and price of fruits using xpath
fruits_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
fruits_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
fruits_names_list = []
quantities = []
prices = []

#getting the current date (without time)
current_date = datetime.now().strftime("%Y-%m-%d")

for i in fruits_names:
    parts = i.text.split(',')
    fruits_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from fruits_price list
for price in fruits_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_fruits.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for fruits_name, qty, price in zip(fruits_names_list, quantities, prices):
        csvwriter.writerow([current_date, fruits_name, qty, price])

# Scraping the vegetable prices

In [6]:
#clicking the Veg & Fruits option from navbar
veg_fruits = driver.find_element(By.XPATH, "//*[contains(text(), 'Veg & Fruits')]").click()

#selecting the fruits from dropdown to be redirected to that webpage
veggies = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-35ae95a75bda48a3-ac60b32fab01d5db']/a[2]"))).click()

In [7]:
# Function to check if all price elements are non-empty
def are_prices_non_empty():
    price_elements = driver.find_elements(By.XPATH, "//div[@class='product-card__price']")
    return all(price_elem.text.strip() for price_elem in price_elements)

#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items

        # Refresh the page until all price elements are non-empty
        while not are_prices_non_empty():
            driver.refresh()
            time.sleep(10)  # adding a delay to ensure the page fully reloads
    except (NoSuchElementException, TimeoutException):
        break

In [8]:
#collecting veg_names and veg_price lists
veg_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
veg_price_min = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
vegetable_names = []
quantities = []
prices = []

for i in veg_names:
    parts = i.text.split(',')
    vegetable_names.append(parts[0].strip())  #first part is Vegetable Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from veg_price_min list
for price in veg_price_min:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_veggies.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for veg_name, qty, price in zip(vegetable_names, quantities, prices):
        csvwriter.writerow([current_date, veg_name, qty, price])
        

# Scraping Dairy Products and Prices

In [9]:
#clicking the Bakery & Dairy option from navbar
dairy = driver.find_element(By.XPATH, "//*[contains(text(), 'Bakery & Dairy')]").click()

#selecting dairy from dropdown to be redirected to that webpage
dairy_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-f8af1b42636542a4-a802ecbe072d3307']/a[2]"))).click()

In [10]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [11]:
#collecting dairy_names and dairy_price lists
dairy_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
dairy_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
dairy_names_list = []
quantities = []
prices = []

for i in dairy_names:
    parts = i.text.split(',')
    dairy_names_list.append(parts[0].strip())  #first part i6*s Dairy Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from dairy_price list
for price in dairy_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_dairy.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for dairy_name, qty, price in zip(dairy_names_list, quantities, prices):
        csvwriter.writerow([current_date, dairy_name, qty, price])

# Scraping Rice Products and Prices

In [12]:
#clicking Grocery option from navbar
grocery = driver.find_element(By.XPATH, "//a[@id='menu-05f8fd1183e14d7e-9e68202448f2de5f']").click()

#selecting rice from dropdown to be redirected to that webpage
rice_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-05f8fd1183e14d7e-9e68202448f2de5f']/a"))).click()

In [13]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [14]:
#collecting rice_names and rice_price lists
rice_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
rice_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
rice_names_list = []
quantities = []
prices = []

for i in rice_names:
    parts = i.text.split(',')
    rice_names_list.append(parts[0].strip())  #first part is rice Name
    quantities.append(parts[-1].strip())  #last part is Quantity

#extracting prices from rice_price list
for price in rice_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_rice.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for rice_name, qty, price in zip(rice_names_list, quantities, prices):
        csvwriter.writerow([current_date, rice_name, qty, price])

# Scraping Atta, Flour and Suji Data

In [15]:
#clicking Grocery option from navbar
grocery = driver.find_element(By.XPATH, "//a[@id='menu-05f8fd1183e14d7e-9e68202448f2de5f']").click()

#selecting option from dropdown to be redirected to that webpage
option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-05f8fd1183e14d7e-9e68202448f2de5f']/a[2]"))).click()

In [16]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [17]:
#collecting names and price lists
product_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
product_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
product_names_list = []
quantities = []
prices = []

for i in product_names:
    parts = i.text.split(',')
    product_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from product_price list
for price in product_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_flour.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for product_name, qty, price in zip(product_names_list, quantities, prices):
        csvwriter.writerow([current_date, product_name, qty, price])

# Scraping Oil Products and Prices

In [18]:
#clicking Grocery option from navbar
grocery = driver.find_element(By.XPATH, "//a[@id='menu-05f8fd1183e14d7e-9e68202448f2de5f']").click()

#selecting option from dropdown to be redirected to that webpage
oil_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-05f8fd1183e14d7e-9e68202448f2de5f']/a[3]"))).click()

In [20]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [ ]:
#collecting names and price lists
oil_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
oil_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
oil_names_list = []
quantities = []
prices = []

for i in oil_names:
    parts = i.text.split(',')
    oil_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from oil_price list
for price in oil_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_oil.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for oil_name, qty, price in zip(oil_names_list, quantities, prices):
        csvwriter.writerow([current_date, oil_name, qty, price])

# Scraping Pulses Products and Prices

In [ ]:
#clicking Grocery option from navbar
grocery = driver.find_element(By.XPATH, "//a[@id='menu-05f8fd1183e14d7e-9e68202448f2de5f']").click()

#selecting option from dropdown to be redirected to that webpage
pulses_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-05f8fd1183e14d7e-9e68202448f2de5f']/a[4]"))).click()

In [ ]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [ ]:
#collecting names and price lists
pulses_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
pulses_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
pulses_names_list = []
quantities = []
prices = []

for i in pulses_names:
    parts = i.text.split(',')
    pulses_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from pulses_price list
for price in pulses_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_pulses.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for pulses_name, qty, price in zip(pulses_names_list, quantities, prices):
        csvwriter.writerow([current_date, pulses_name, qty, price])

# Scraping Dry Fruits Products and Prices

In [ ]:
#clicking Grocery option from navbar
grocery = driver.find_element(By.XPATH, "//a[@id='menu-05f8fd1183e14d7e-9e68202448f2de5f']").click()

#selecting option from dropdown to be redirected to that webpage
dry_fruits_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-05f8fd1183e14d7e-9e68202448f2de5f']/a[5]"))).click()

In [ ]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [ ]:
#collecting dry_fruits_names and dry_fruits_price lists
dry_fruits_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
dry_fruits_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
dry_fruits_names_list = []
quantities = []
prices = []

for i in dry_fruits_names:
    parts = i.text.split(',')
    dry_fruits_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from dry_fruits_price list
for price in dry_fruits_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_dry_fruits.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for dry_fruits_name, qty, price in zip(dry_fruits_names_list, quantities, prices):
        csvwriter.writerow([current_date, dry_fruits_name, qty, price])

# Scraping Salt and Sugar Products and Prices

In [ ]:
#clicking Grocery option from navbar
grocery = driver.find_element(By.XPATH, "//a[@id='menu-05f8fd1183e14d7e-9e68202448f2de5f']").click()

#selecting option from dropdown to be redirected to that webpage
salt_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-05f8fd1183e14d7e-9e68202448f2de5f']/a[7]"))).click()

In [ ]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [ ]:
#collecting the name and price of salt using xpath
salt_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
salt_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
salt_names_list = []
quantities = []
prices = []

for i in salt_names:
    parts = i.text.split(',')
    salt_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from salt_price list
for price in salt_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_salt.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for salt_name, qty, price in zip(salt_names_list, quantities, prices):
        csvwriter.writerow([current_date, salt_name, qty, price])

# Scraping Spices Products and Prices

In [ ]:
#clicking Grocery option from navbar
grocery = driver.find_element(By.XPATH, "//a[@id='menu-05f8fd1183e14d7e-9e68202448f2de5f']").click()

#selecting option from dropdown to be redirect"ed to that webpage
spices_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-05f8fd1183e14d7e-9e68202448f2de5f']/a[8]"))).click()

In [ ]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [ ]:
#collecting spices_names and spices_price lists
spices_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
spices_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
spices_names_list = []
quantities = []
prices = []

for i in spices_names:
    parts = i.text.split(',')
    spices_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from spices_price list
for price in spices_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_spices.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for spices_name, qty, price in zip(spices_names_list, quantities, prices):
        csvwriter.writerow([current_date, spices_name, qty, price])

# Scraping Eggs Products and Prices

In [ ]:
#clicking Eggs & Meat option from navbar
eggs = driver.find_element(By.XPATH, "//a[@id='menu-8ac3e639f1224f43-999c081adb4d0794']").click()

#selecting option from dropdown to be redirected to that webpage
eggs_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-8ac3e639f1224f43-999c081adb4d0794']/a"))).click()

In [ ]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [ ]:
#collecting the name and price of eggs using xpath
egg_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
egg_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
egg_names_list = []
quantities = []
prices = []

for i in egg_names:
    parts = i.text.split(',')
    egg_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from egg_price list
for price in egg_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_eggs.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for egg_name, qty, price in zip(egg_names_list, quantities, prices):
        csvwriter.writerow([current_date, egg_name, qty, price])

# Scraping Meat Products and Prices

In [ ]:
#clicking Eggs & Meat option from navbar
meats = driver.find_element(By.XPATH, "//a[@id='menu-8ac3e639f1224f43-999c081adb4d0794']").click()

#selecting option from dropdown to be redirected to that webpage
meat_option = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-8ac3e639f1224f43-999c081adb4d0794']/a[2]"))).click()

In [ ]:
#clicking the "Load More" button and waiting until the button is no longer clickable
while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
        )
        load_more_button.click()
        time.sleep(2)  # adding a delay to ensure that the page loads the new items
    except (NoSuchElementException, TimeoutException):
        break

# Wait for any onsale product cards to be clickable before scraping
try:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
    )
except TimeoutException:
    print("No onsale product cards are clickable.")

In [ ]:
#collecting meat_names and meat_price lists
meat_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
meat_price = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

#creating lists to store data
meat_names_list = []
quantities = []
prices = []

for i in meat_names:
    parts = i.text.split(',')
    meat_names_list.append(parts[0].strip())  #first part is Name
    quantities.append(parts[-1].strip())  #last part is Quantity
    
#extracting prices from meat_price list
for price in meat_price:
    prices.append(price.text.strip())  #extracting prices

#writing data to CSV file
with open('MK_meat.csv', 'a', newline='', encoding='utf-8') as csvfile:
    csvwriter = csv.writer(csvfile)
#     csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

    #writing data into CSV file
    #using zip() to pair the corresponding values into tuples
    for meat_name, qty, price in zip(meat_names_list, quantities, prices):
        csvwriter.writerow([current_date, meat_name, qty, price])

In [ ]:
#closing the WebDriver instance
driver.quit()

In [ ]:
# #importing libraries
# import csv
# import time
# from datetime import datetime
# from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.common.exceptions import NoSuchElementException, TimeoutException, StaleElementReferenceException

# #specifying the path to the Chrome binary
# chrome_binary_path = "C:\Program Files\Google\Chrome Beta\Application\chrome.exe"

# #setting the binary location in ChromeOptions
# chrome_options = webdriver.ChromeOptions()
# chrome_options.binary_location = chrome_binary_path

# #specifying the path to ChromeDriver
# chrome_driver_path = r"D:\chromedriver-win64\chromedriver.exe"

# #creating a ChromeDriver service with the specified binary path
# service = Service(executable_path=chrome_driver_path)

# #creating the WebDriver instance with the specified options and service
# driver = webdriver.Chrome(service=service, options=chrome_options)

# #maximizing the window and opening the link for scraping
# driver.maximize_window()
# driver.get('https://merokirana.com/#/')

# #clicking the Veg & Fruits option from navbar
# veg_fruits = driver.find_element(By.XPATH, "//*[contains(text(), 'Veg & Fruits')]").click()

# #selecting the fruits from dropdown to be redirected to that webpage
# fruits = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-35ae95a75bda48a3-ac60b32fab01d5db']/a"))).click()

# # Function to check if all price elements are non-empty
# def are_prices_non_empty():
#     price_elements = driver.find_elements(By.XPATH, "//div[@class='product-card__price']")
#     return all(price_elem.text.strip() for price_elem in price_elements)

# #clicking the "Load More" button and waiting until the button is no longer clickable
# while True:
#     try:
#         load_more_button = WebDriverWait(driver, 10).until(
#             EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
#         )
#         load_more_button.click()
#         time.sleep(2)  # adding a delay to ensure that the page loads the new items
#     except (NoSuchElementException, TimeoutException):
#         break

# # Wait for any onsale product cards to be clickable before scraping
# try:
#     WebDriverWait(driver, 10).until(
#         EC.element_to_be_clickable((By.XPATH, "//div[@class='product-card product-card--onsale']"))
#     )
# except TimeoutException:
#     print("No onsale product cards are clickable.")

# # Refresh the page until all price elements are non-empty
# while not are_prices_non_empty():
#     driver.refresh()
#     time.sleep(5)  # adding a delay to ensure the page fully reloads

# #collecting the name, quantity, and price of fruits using xpath
# fruits_elements = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
# price_elements = driver.find_elements(By.XPATH, "//div[@class='product-card__price']")

# #creating lists to store data
# fruits_names_list = []
# quantities = []
# prices = []

# #getting the current date (without time)
# current_date = datetime.now().strftime("%Y-%m-%d")

# for fruit_elem, price_elem in zip(fruits_elements, price_elements):
#     # Since we have ensured all price elements are non-empty, we can proceed to scrape
#     parts = fruit_elem.text.split(',')
#     fruits_names_list.append(parts[0].strip())  #first part is Name
#     quantities.append(parts[-1].strip())  #last part is Quantity
#     prices.append(price_elem.text.strip())  #extracting prices

# #writing data to CSV file
# with open('MK_fruits.csv', 'a', newline='', encoding='utf-8') as csvfile:
#     csvwriter = csv.writer(csvfile)
#     # csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

#     #writing data into CSV file
#     #using zip() to pair the corresponding values into tuples
#     for fruits_name, qty, price in zip(fruits_names_list, quantities, prices):
#         csvwriter.writerow([current_date, fruits_name, qty, price])

# #closing the WebDriver instance
# driver.quit()

In [ ]:
# #importing libraries
# import csv
# import time
# from datetime import datetime
# from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.common.exceptions import NoSuchElementException, TimeoutException, StaleElementReferenceException

# #specifying the path to the Chrome binary
# chrome_binary_path = "C:\Program Files\Google\Chrome Beta\Application\chrome.exe"

# #setting the binary location in ChromeOptions
# chrome_options = webdriver.ChromeOptions()
# chrome_options.binary_location = chrome_binary_path

# #specifying the path to ChromeDriver
# chrome_driver_path = r"D:\chromedriver-win64\chromedriver.exe"

# #creating a ChromeDriver service with the specified binary path
# service = Service(executable_path=chrome_driver_path)

# #creating the WebDriver instance with the specified options and service
# driver = webdriver.Chrome(service=service, options=chrome_options)

# #maximizing the window and opening the link for scraping
# driver.maximize_window()
# driver.get('https://merokirana.com/#/')

# #clicking the Veg & Fruits option from navbar
# veg_fruits = driver.find_element(By.XPATH, "//*[contains(text(), 'Veg & Fruits')]").click()

# #selecting the veggies from dropdown to be redirected to that webpage
# veggies = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//div[@id='inner-35ae95a75bda48a3-ac60b32fab01d5db']/a[2]"))).click()

# # Function to check if all price elements are non-empty
# def are_prices_non_empty():
#     price_elements = driver.find_elements(By.XPATH, "//div[@class='product-card__price']")
#     return all(price_elem.text.strip() for price_elem in price_elements)

# #clicking the "Load More" button and waiting until the button is no longer clickable
# while True:
#     try:
#         load_more_button = WebDriverWait(driver, 10).until(
#             EC.element_to_be_clickable((By.XPATH, "//button[@title='Load More ']"))
#         )
#         load_more_button.click()
#         time.sleep(2)  # adding a delay to ensure that the page loads the new items

#         # Refresh the page until all price elements are non-empty
#         while not are_prices_non_empty():
#             driver.refresh()
#             time.sleep(10)  # adding a delay to ensure the page fully reloads
#     except (NoSuchElementException, TimeoutException):
#         break

# #collecting veg_names and veg_price lists
# veg_names = driver.find_elements(By.XPATH, "//div[@class='product-card__info']/h2/a")
# veg_price_min = driver.find_elements(By.XPATH, "//span[@class='product-card__actual-price']/span[2]")

# #creating lists to store data
# vegetable_names = []
# quantities = []
# prices = []

# #getting the current date (without time)
# current_date = datetime.now().strftime("%Y-%m-%d")

# for veg_elem, price_elem in zip(veg_names, veg_price_min):
#     # Check if price element is not empty
#     if price_elem.text.strip():
#         # Get the vegetable name and quantity
#         parts = veg_elem.text.split(',')
#         vegetable_names.append(parts[0].strip())  #first part is Vegetable Name
#         quantities.append(parts[-1].strip())  #last part is Quantity
#         # Get the price
#         prices.append(price_elem.text.strip())  #extracting prices

# #writing data to CSV file
# with open('MK_veggies.csv', 'a', newline='', encoding='utf-8') as csvfile:
#     csvwriter = csv.writer(csvfile)
#     # csvwriter.writerow(['Date', 'Product Name', 'Quantity', 'Price'])  #writing header row

#     #writing data into CSV file
#     #using zip() to pair the corresponding values into tuples
#     for veg_name, qty, price in zip(vegetable_names, quantities, prices):
#         csvwriter.writerow([current_date, veg_name, qty, price])

# #closing the WebDriver instance
# driver.quit()